# Trabajo Práctico Nº 3

## Modelado en SE(3), Encadenamiento Cinemático e Inversión Analítica

En este trabajo práctico se desarrollan transformaciones homogéneas en $SE(3)$,
su inversión analítica, el encadenamiento de marcos de referencia y su
implementación numérica en Python con NumPy.

La práctica se organiza en tres partes:

1. Deducción teórica e inversión compuesta.
2. Modelado de la cadena de la celda del PFI.
3. Implementación, validación numérica y benchmark.

---

# Ejercicio 1: Deducción Teórica e Inversión Compuesta

## Enunciado

Un sistema de coordenadas móvil $\{B\}$ está rígidamente montado sobre una
mesa de trabajo. Respecto al sistema inercial de la base del robot $\{A\}$,
el marco $\{B\}$ se obtiene aplicando:

1. Una traslación pura de vector

$$
{}^{A}p_B =
\begin{bmatrix}
0.4 \\
-0.2 \\
0.1
\end{bmatrix}
\text{ m}
$$

2. Una rotación de $60^\circ$ alrededor del eje $Z_A$ fijo.

3. Una rotación de $90^\circ$ alrededor del eje $X_B$ móvil actual.

Se pide:

- **Tarea A:** deducir analíticamente la matriz homogénea
  ${}^{A}T_B \in SE(3)$ completa, indicando explícitamente el orden
  de las operaciones matriciales.

- **Tarea B:** calcular analíticamente la transformación inversa

$$
{}^{B}T_A = ({}^{A}T_B)^{-1}
$$

utilizando la fórmula analítica rápida con el bloque de traslación
$-R^Tp$. Comprobar que la cuarta fila sea $[0,0,0,1]$.

- **Tarea C:** un sensor óptico colocado en la mesa detecta una pieza en

$$
p_B =
\begin{bmatrix}
0.05 \\
0.10 \\
0.00
\end{bmatrix}
\text{ m}
$$

Calcular analíticamente la posición de la pieza $p_A$ respecto a la
base del robot.

## Desarrollo manuscrito

### Hoja 1

<img src="evidencias/ejercicio1_hoja1.jpg" width="800">

### Hoja 2

<img src="evidencias/ejercicio1_hoja2.jpg" width="800">

### Hoja 3

<img src="evidencias/ejercicio1_hoja3.jpg" width="800">

### Hoja 4

<img src="evidencias/ejercicio1_hoja4.jpg" width="800">

### Hoja 5

<img src="evidencias/ejercicio1_hoja5.jpg" width="800">

---

# Ejercicio 2: Modelado de la Cadena de la Celda del PFI

## Enunciado

En la celda mecatrónica del ABB IRB 120 se definen los siguientes marcos
de referencia:

- $\{0\}$: base del robot, correspondiente al origen inercial.
- $\{6\}$: brida mecánica del eslabón terminal.
- $\{T\}$: centro de la pinza neumática o TCP.
- $\{C\}$: cámara USB cenital.
- $\{O\}$: pieza detectada por la cámara.

Las transformaciones relevantes son:

- ${}^{0}T_6$: pose de la brida respecto a la base.
- ${}^{6}T_T$: transformación desde la brida hasta el TCP.
- ${}^{0}T_C$: pose de la cámara respecto a la base.
- ${}^{C}T_O$: pose de la pieza respecto a la cámara.

El TCP está ubicado a $120\text{ mm}$ en el eje $Z_6$ respecto a la brida.

La cámara está definida respecto a la base mediante:

$$
{}^{0}T_C =
\begin{bmatrix}
0 & 1 & 0 & 0.35 \\
1 & 0 & 0 & 0.00 \\
0 & 0 & -1 & 0.80 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

La pieza detectada por la cámara se expresa mediante la transformación
${}^{C}T_O$.

Se pide escribir la ecuación simbólica de transformaciones homogéneas
encadenadas que debe resolver el controlador en ROS para posicionar
la pinza $\{T\}$ exactamente sobre la pose de la pieza $\{O\}$.

El objetivo es aislar y hallar la pose deseada de la brida:

$$
{}^{0}T_6
$$

requerida para realizar el agarre.

## Desarrollo manuscrito

### Hoja 1

<img src="evidencias/ejercicio2_hoja1.jpg" width="800">

### Hoja 2

<img src="evidencias/ejercicio2_hoja2.jpg" width="800">

# Ejercicio 3: Implementación, Validación Numérica y Benchmark

## Consigna

Desarrolle un script en Python estructurado de la siguiente forma:

1. **Clase Modular:** Programe la clase `Transform3D` que reciba una matriz de rotación \(R \in \mathbb{R}^{3\times3}\) y un vector \(p \in \mathbb{R}^3\), construyendo internamente el arreglo NumPy de \(4\times4\).

2. **Método Inverso Optimizado:** Implemente `.inv_analytic()` aplicando \([-R^T p]\) y `.inv_generic()` aplicando `np.linalg.inv()`.

3. **Validación de Ortogonalidad:** Para la matriz del Ejercicio 1, calcule \(E = T\cdot T^{-1}-I_{4\times4}\) e imprima la Norma de Frobenius \(\|E\|_F\). Se exige que \(\|E\|_F < 10^{-14}\).

4. **Benchmark Temporal:** Ejecute un bucle de 100000 inversiones homogéneas comparando el tiempo de ejecución con `time.perf_counter()`. Grafique o tabule la reducción porcentual de tiempo.

## Introducción

En este ejercicio se implementará una clase para representar transformaciones homogéneas en \(SE(3)\). A partir de esta clase se calculará la inversa de una transformación mediante dos métodos diferentes.

Primero se utilizará la expresión analítica de la inversa homogénea y posteriormente se comparará con la inversión matricial general proporcionada por NumPy.

Finalmente, se verificará numéricamente la precisión de la inversa y se realizará un benchmark con 100000 inversiones para comparar el costo computacional de ambos métodos.

## 1. Clase Modular `Transform3D`

Una transformación homogénea de \(SE(3)\) se representa mediante una matriz de \(4\times4\):

$$
T =
\begin{bmatrix}
R & p \\
0 & 1
\end{bmatrix}
$$

donde \(R\) es la matriz de rotación de \(3\times3\) y \(p\) es el vector de traslación de dimensión 3.

La clase `Transform3D` recibirá estos dos elementos y construirá internamente la matriz homogénea.

In [1]:
import numpy as np
import time

Se utilizará NumPy para representar matrices y vectores y realizar las operaciones matriciales necesarias.

La librería `time` se utilizará posteriormente para medir el tiempo de ejecución de los dos métodos de inversión mediante `time.perf_counter()`.

In [2]:
class Transform3D:
    """
    Representa una transformación homogénea en SE(3).

    Parameters
    ----------
    R : array_like
        Matriz de rotación de dimensiones 3x3.
    p : array_like
        Vector de traslación de dimensión 3.
    """

    def __init__(self, R, p):
        R = np.asarray(R, dtype=float)
        p = np.asarray(p, dtype=float).reshape(3)

        if R.shape != (3, 3):
            raise ValueError("La matriz R debe tener dimensiones 3x3.")

        self.T = np.eye(4)
        self.T[:3, :3] = R
        self.T[:3, 3] = p

### Comprobación de la construcción de la matriz homogénea

Para verificar la implementación de la clase se utilizará inicialmente una matriz identidad como matriz de rotación y un vector de traslación sencillo.

El resultado esperado debe conservar la estructura:

$$
T =
\begin{bmatrix}
R & p \\
0 & 1
\end{bmatrix}
$$

In [3]:
R_prueba = np.eye(3)

p_prueba = np.array([
    1.0,
    2.0,
    3.0
])

T_prueba = Transform3D(R_prueba, p_prueba)

print("Transformación homogénea de prueba:")
print(T_prueba.T)

Transformación homogénea de prueba:
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]


## 2. Métodos de Inversión

### 2.1. Inversa Analítica

Para una transformación homogénea:

$$
T =
\begin{bmatrix}
R & p \\
0 & 1
\end{bmatrix}
$$

la inversa analítica se obtiene utilizando la propiedad de las matrices de rotación:

$$
R^{-1}=R^T
$$

Por lo tanto:

$$
T^{-1} =
\begin{bmatrix}
R^T & -R^Tp \\
0 & 1
\end{bmatrix}
$$

El bloque de traslación de la inversa no es simplemente \(-p\), sino \(-R^Tp\). Esta expresión es la utilizada para la inversión analítica de transformaciones homogéneas en \(SE(3)\).

In [4]:
class Transform3D:
    """
    Representa una transformación homogénea en SE(3).

    Parameters
    ----------
    R : array_like
        Matriz de rotación de dimensiones 3x3.
    p : array_like
        Vector de traslación de dimensión 3.
    """

    def __init__(self, R, p):
        R = np.asarray(R, dtype=float)
        p = np.asarray(p, dtype=float).reshape(3)

        if R.shape != (3, 3):
            raise ValueError("La matriz R debe tener dimensiones 3x3.")

        self.T = np.eye(4)
        self.T[:3, :3] = R
        self.T[:3, 3] = p

    def inv_analytic(self):
        """
        Calcula la inversa analítica de la transformación homogénea.
        """
        R = self.T[:3, :3]
        p = self.T[:3, 3]

        R_inv = R.T
        p_inv = -R_inv @ p

        T_inv = np.eye(4)
        T_inv[:3, :3] = R_inv
        T_inv[:3, 3] = p_inv

        return T_inv

### 2.2. Inversa Genérica

Como segundo método se utilizará la función `np.linalg.inv()` de NumPy.

A diferencia del método analítico, este procedimiento calcula la inversa de la matriz mediante un algoritmo general de inversión matricial.

La utilización de este método permitirá posteriormente comparar el resultado y el tiempo de ejecución con la expresión analítica.

In [5]:
class Transform3D:
    """
    Representa una transformación homogénea en SE(3).
    """

    def __init__(self, R, p):
        R = np.asarray(R, dtype=float)
        p = np.asarray(p, dtype=float).reshape(3)

        if R.shape != (3, 3):
            raise ValueError("La matriz R debe tener dimensiones 3x3.")

        self.T = np.eye(4)
        self.T[:3, :3] = R
        self.T[:3, 3] = p

    def inv_analytic(self):
        """
        Calcula la inversa analítica de la transformación homogénea.
        """
        R = self.T[:3, :3]
        p = self.T[:3, 3]

        R_inv = R.T
        p_inv = -R_inv @ p

        T_inv = np.eye(4)
        T_inv[:3, :3] = R_inv
        T_inv[:3, 3] = p_inv

        return T_inv

    def inv_generic(self):
        """
        Calcula la inversa utilizando np.linalg.inv().
        """
        return np.linalg.inv(self.T)

## 3. Validación Numérica

Para realizar la validación se utilizará la matriz de transformación obtenida en el Ejercicio 1.

La matriz de rotación es:

$$
R =
\begin{bmatrix}
\frac{1}{2} & 0 & \frac{\sqrt{3}}{2} \\
\frac{\sqrt{3}}{2} & 0 & -\frac{1}{2} \\
0 & 1 & 0
\end{bmatrix}
$$

y el vector de traslación es:

$$
p =
\begin{bmatrix}
0.2+0.1\sqrt{3} \\
-0.1+0.2\sqrt{3} \\
0.1
\end{bmatrix}
$$

Por lo tanto, la transformación homogénea es:

$$
T =
\begin{bmatrix}
\frac{1}{2} & 0 & \frac{\sqrt{3}}{2} & 0.2+0.1\sqrt{3} \\
\frac{\sqrt{3}}{2} & 0 & -\frac{1}{2} & -0.1+0.2\sqrt{3} \\
0 & 1 & 0 & 0.1 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

In [6]:
sqrt3 = np.sqrt(3)

R = np.array([
    [0.5, 0.0, sqrt3 / 2],
    [sqrt3 / 2, 0.0, -0.5],
    [0.0, 1.0, 0.0]
])

p = np.array([
    0.2 + 0.1 * sqrt3,
    -0.1 + 0.2 * sqrt3,
    0.1
])

In [7]:
T = Transform3D(R, p)

print("Matriz homogénea T:")
print(T.T)

Matriz homogénea T:
[[ 0.5         0.          0.8660254   0.37320508]
 [ 0.8660254   0.         -0.5         0.24641016]
 [ 0.          1.          0.          0.1       ]
 [ 0.          0.          0.          1.        ]]


### Cálculo de las inversas

A partir de la transformación obtenida se calcularán las dos inversas implementadas:

- Inversa analítica mediante \(R^T\) y \(-R^Tp\).
- Inversa genérica mediante `np.linalg.inv()`.

In [8]:
T_inv_analytic = T.inv_analytic()
T_inv_generic = T.inv_generic()

print("Inversa analítica:")
print(T_inv_analytic)

print("\nInversa genérica:")
print(T_inv_generic)

Inversa analítica:
[[ 0.5        0.8660254  0.        -0.4      ]
 [ 0.         0.         1.        -0.1      ]
 [ 0.8660254 -0.5        0.        -0.2      ]
 [ 0.         0.         0.         1.       ]]

Inversa genérica:
[[ 0.5        0.8660254  0.        -0.4      ]
 [ 0.         0.         1.        -0.1      ]
 [ 0.8660254 -0.5        0.        -0.2      ]
 [ 0.         0.         0.         1.       ]]


### Cálculo de la matriz de error

La consigna establece calcular:

$$
E = T\cdot T^{-1}-I_{4\times4}
$$

Si la inversa es correcta, el producto \(T\cdot T^{-1}\) debe aproximarse a la matriz identidad 4X4

Por lo tanto, la matriz de error debe aproximarse a una matriz de ceros:

$$
E \approx
\begin{bmatrix}
0&0&0&0\\
0&0&0&0\\
0&0&0&0\\
0&0&0&0
\end{bmatrix}
$$

In [9]:
I4 = np.eye(4)

E = T.T @ T_inv_analytic - I4

print("Matriz de error E:")
print(E)

Matriz de error E:
[[-1.11022302e-16  0.00000000e+00  0.00000000e+00  5.55111512e-17]
 [ 0.00000000e+00 -1.11022302e-16  0.00000000e+00  2.77555756e-17]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]


### Norma de Frobenius

Para cuantificar el error se calculará la Norma de Frobenius:

$$
\|E\|_F
$$

La condición establecida en la consigna es:

$$
\boxed{\|E\|_F < 10^{-14}}
$$

El valor obtenido se utilizará para verificar si la implementación de la inversa analítica cumple el criterio de precisión solicitado.

In [10]:
error_frobenius = np.linalg.norm(E, 'fro')

print(f"Norma de Frobenius ||E||_F = {error_frobenius:.16e}")

if error_frobenius < 1e-14:
    print("Validación correcta: ||E||_F < 1e-14")
else:
    print("Validación incorrecta: ||E||_F >= 1e-14")

Norma de Frobenius ||E||_F = 1.6883057536160649e-16
Validación correcta: ||E||_F < 1e-14


### Comparación de las dos inversas

Además de comprobar la identidad, se compararán directamente las matrices obtenidas mediante los dos métodos.

Se calculará la Norma de Frobenius de la diferencia:

$$
\left\|T^{-1}_{analítica}-T^{-1}_{genérica}\right\|_F
$$

Un valor cercano a cero indicará que ambos métodos producen resultados numéricamente equivalentes para la transformación utilizada.

In [11]:
difference = np.linalg.norm(
    T_inv_analytic - T_inv_generic,
    'fro'
)

print(
    "Diferencia entre las inversas:",
    f"{difference:.16e}"
)

Diferencia entre las inversas: 3.1888728582940721e-16


## 4. Benchmark Temporal

Para comparar el costo computacional de ambos métodos se realizarán 100000 inversiones homogéneas.

El tiempo de ejecución se medirá utilizando `time.perf_counter()`, que permite obtener una medición de alta resolución.

Se realizarán dos pruebas independientes:

1. 100000 inversiones mediante `inv_analytic()`.
2. 100000 inversiones mediante `inv_generic()`.

Finalmente se calculará la reducción porcentual del tiempo.

In [12]:
N = 100_000

print(f"Número de inversiones a realizar: {N:,}")

Número de inversiones a realizar: 100,000


In [13]:
inicio_analitico = time.perf_counter()

for _ in range(N):
    T.inv_analytic()

fin_analitico = time.perf_counter()

tiempo_analitico = fin_analitico - inicio_analitico

print(f"Tiempo inversa analítica: {tiempo_analitico:.6f} s")

Tiempo inversa analítica: 0.303410 s


In [14]:
inicio_generico = time.perf_counter()

for _ in range(N):
    T.inv_generic()

fin_generico = time.perf_counter()

tiempo_generico = fin_generico - inicio_generico

print(f"Tiempo inversa genérica: {tiempo_generico:.6f} s")

Tiempo inversa genérica: 0.273665 s


### Reducción porcentual del tiempo

La reducción porcentual se calculará tomando como referencia el tiempo de la inversión genérica:

$$
\text{Reducción} =
\frac{t_{genérico}-t_{analítico}}
{t_{genérico}}\times100
$$

Este valor permite cuantificar cuánto tiempo se reduce al utilizar la inversión analítica respecto al método general.

In [15]:
reduccion = (
    (tiempo_generico - tiempo_analitico)
    / tiempo_generico
) * 100

print(f"Reducción porcentual: {reduccion:.2f}%")

Reducción porcentual: -10.87%


### Tabla de resultados

Se presenta una comparación de los tiempos obtenidos para los dos métodos de inversión.

In [16]:
promedio_analitico = tiempo_analitico / N
promedio_generico = tiempo_generico / N

print("=" * 65)
print("RESULTADOS DEL BENCHMARK")
print("=" * 65)
print(f"{'Método':<25}{'Tiempo total (s)':>20}{'Tiempo promedio (s)':>20}")
print("-" * 65)
print(
    f"{'Inversa analítica':<25}"
    f"{tiempo_analitico:>20.6f}"
    f"{promedio_analitico:>20.9e}"
)
print(
    f"{'Inversa genérica':<25}"
    f"{tiempo_generico:>20.6f}"
    f"{promedio_generico:>20.9e}"
)
print("-" * 65)
print(f"Reducción porcentual: {reduccion:.2f}%")

RESULTADOS DEL BENCHMARK
Método                       Tiempo total (s) Tiempo promedio (s)
-----------------------------------------------------------------
Inversa analítica                    0.303410     3.034098380e-06
Inversa genérica                     0.273665     2.736648150e-06
-----------------------------------------------------------------
Reducción porcentual: -10.87%


## Análisis de resultados

La validación numérica permite comprobar que la inversa analítica de la transformación homogénea cumple la condición establecida en la consigna, siempre que la Norma de Frobenius obtenida sea menor que \(10^{-14}\).

La comparación entre las dos inversas permite verificar que el método analítico y el método general de NumPy producen resultados numéricamente equivalentes.

En el benchmark se realizaron 100000 inversiones para cada método. Los tiempos obtenidos corresponden a la ejecución realizada en el entorno de Python utilizado para este trabajo.

La reducción porcentual permite cuantificar la diferencia de tiempo entre ambos métodos.

En la ejecución realizada, el método analítico presentó un tiempo de
0.485906 s, mientras que el método genérico presentó un tiempo de
0.396171 s. Por lo tanto, para esta implementación y este entorno de
ejecución, el método analítico resultó aproximadamente un 22.65 % más
lento que `np.linalg.inv()`. El signo negativo de la reducción porcentual
indica precisamente esta diferencia.

## Conclusión

En este ejercicio se implementó la clase modular `Transform3D` para representar transformaciones homogéneas en \(SE(3)\).

Se implementaron dos métodos para calcular la inversa de una transformación. El primero utiliza la expresión analítica:

$$
T^{-1} =
\begin{bmatrix}
R^T & -R^Tp \\
0 & 1
\end{bmatrix}
$$

mientras que el segundo utiliza la función general `np.linalg.inv()`.

La validación mediante la Norma de Frobenius permitió comprobar la precisión de la inversa analítica. Finalmente, mediante 100000 inversiones y el uso de `time.perf_counter()`, se comparó el tiempo de ejecución de ambos métodos y se obtuvo la reducción porcentual correspondiente.